In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("load_type", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = (dbutils.widgets.get("load_type") or "FULL").strip().upper()
merge_key_list = [key.strip() for key in merge_keys.split(",") if key.strip()]

print("=" * 80)
print("Step 1: Parameters received")
print(f"landing_timestamp = {landing_timestamp}")
print(f"table_name        = {table_name}")
print(f"load_type         = {load_type}")
print(f"merge_keys raw    = {merge_keys}")
print(f"merge_key_list    = {merge_key_list}")
print("=" * 80)

In [0]:
source_table = f"nildbkws.bronze.{table_name}"
source_query = f"SELECT * FROM {source_table} WHERE landing_timestamp = '{landing_timestamp}'"

print("Step 2: Reading source data from bronze")
print(f"Source table : {source_table}")
print("Source query :")
print(source_query)

bronze_df = spark.sql(source_query)
source_count = bronze_df.count()
print(f"Source record count for landing_timestamp '{landing_timestamp}' = {source_count}")


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

print("Step 3: Preparing dataframe for silver load")
print("Adding audit columns: insert_timestamp, updated_timestamp")

updated_df = bronze_df.withColumn("insert_timestamp", F.current_timestamp()) \
    .withColumn("updated_timestamp", F.current_timestamp())

print("Prepared dataframe columns:")
print(updated_df.columns)


In [0]:
target_table = f"nildbkws.silver.{table_name}"

target_exists = spark.catalog.tableExists(target_table)
print("Step 4: Loading data into silver")
print(f"Target table           : {target_table}")
print(f"Target table exists    : {target_exists}")
print(f"Selected load_type     : {load_type}")

if load_type == "FULL":
    full_load_query = f"INSERT OVERWRITE {target_table} SELECT * FROM <prepared_source_dataframe>"
    print("Executing FULL load")
    print("Prepared write operation :")
    print(full_load_query)
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("FULL load completed successfully")
elif load_type == "APPEND":
    append_query = f"INSERT INTO {target_table} SELECT * FROM <prepared_source_dataframe>"
    print("Executing APPEND load")
    print("Prepared write operation :")
    print(append_query)
    updated_df.write.mode("append").saveAsTable(target_table)
    print("APPEND load completed successfully")
elif load_type == "MERGE":
    if not merge_key_list:
        raise ValueError("merge_keys must contain one or more comma-separated columns when load_type is MERGE.")

    print("Executing MERGE load")
    print(f"Merge keys             : {merge_key_list}")

    if target_exists:
        merge_condition = " AND ".join([f"target.`{column}` = source.`{column}`" for column in merge_key_list])
        update_map = {column: f"source.`{column}`" for column in updated_df.columns if column != "insert_timestamp"}
        merge_set_clause = ",\n    ".join([f"target.`{column}` = source.`{column}`" for column in update_map.keys()])
        insert_columns = ", ".join([f"`{column}`" for column in updated_df.columns])
        insert_values = ", ".join([f"source.`{column}`" for column in updated_df.columns])
        merge_query = f"""MERGE INTO {target_table} AS target
USING <prepared_source_dataframe> AS source
ON {merge_condition}
WHEN MATCHED THEN UPDATE SET
    {merge_set_clause}
WHEN NOT MATCHED THEN INSERT ({insert_columns})
VALUES ({insert_values})"""

        print("Prepared merge query :")
        print(merge_query)

        DeltaTable.forName(spark, target_table).alias("target") \
            .merge(updated_df.alias("source"), merge_condition) \
            .whenMatchedUpdate(set=update_map) \
            .whenNotMatchedInsertAll() \
            .execute()
        print("MERGE load completed successfully")
    else:
        print("Target table does not exist. Creating target table with initial load.")
        initial_merge_query = f"CREATE TABLE {target_table} AS SELECT * FROM <prepared_source_dataframe>"
        print("Prepared initial load operation :")
        print(initial_merge_query)
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print("Initial target table creation completed successfully")
else:
    raise ValueError("load_type must be one of FULL, APPEND, or MERGE.")

In [0]:
print("Step 5: Calculating processed record count")
record_count = updated_df.count()
print(f"Processed record count = {record_count}")
print("Notebook execution completed successfully")

dbutils.notebook.exit(str(record_count))
